# CSC 5800 - Intelligent Systems: Algorithms and Tools
## Project: Hybrid Network Intrusion Detection System
### Notebook 1: Environment Setup & Data Acquisition
**Student:** Fahad Qaseem Khawar  
**Instructor:** Dr. Suzan Arslanturk  
**Semester:** Winter 2026

---
This notebook handles:
- Mounting Google Drive
- Creating the project folder structure
- Installing required libraries
- Downloading the UNSW-NB15 dataset
- Verifying everything is in place

## Step 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully.')

Mounted at /content/drive
Drive mounted successfully.


## Step 2: Create Project Folder Structure

In [2]:
import os

BASE = '/content/drive/MyDrive/CSC5800_NIDS_Project'

folders = [
    BASE,
    f'{BASE}/data/raw',
    f'{BASE}/data/processed',
    f'{BASE}/models',
    f'{BASE}/results',
    f'{BASE}/figures',
    f'{BASE}/notebooks',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('\nProject folder structure ready.')

Created: /content/drive/MyDrive/CSC5800_NIDS_Project
Created: /content/drive/MyDrive/CSC5800_NIDS_Project/data/raw
Created: /content/drive/MyDrive/CSC5800_NIDS_Project/data/processed
Created: /content/drive/MyDrive/CSC5800_NIDS_Project/models
Created: /content/drive/MyDrive/CSC5800_NIDS_Project/results
Created: /content/drive/MyDrive/CSC5800_NIDS_Project/figures
Created: /content/drive/MyDrive/CSC5800_NIDS_Project/notebooks

Project folder structure ready.


## Step 3: Install Required Libraries

In [3]:
!pip install mlxtend xgboost lightgbm imbalanced-learn -q
print('All libraries installed.')

All libraries installed.


## Step 4: Verify Library Imports

In [4]:
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import xgboost
import lightgbm
import mlxtend
import imblearn

libs = {
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'matplotlib': matplotlib.__version__,
    'scikit-learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
    'lightgbm': lightgbm.__version__,
    'mlxtend': mlxtend.__version__,
    'imbalanced-learn': imblearn.__version__,
}

print('Library versions:')
for lib, ver in libs.items():
    print(f'  {lib}: {ver}')

Library versions:
  numpy: 2.0.2
  pandas: 2.2.2
  matplotlib: 3.10.0
  scikit-learn: 1.6.1
  xgboost: 3.2.0
  lightgbm: 4.6.0
  mlxtend: 0.23.4
  imbalanced-learn: 0.14.1


## Step 5: Download UNSW-NB15 Dataset

The UNSW-NB15 dataset is hosted by the University of New South Wales.  
We download the four CSV partitions and the features list file.

In [5]:
import urllib.request

RAW = f'{BASE}/data/raw'

files = {
    'UNSW-NB15_1.csv': 'https://cloudstor.aarnet.edu.au/plus/s/2DhnLGDdEECo4ys/download?path=%2FUNSW-NB15%20-%20CSV%20Files&files=UNSW-NB15_1.csv',
    'UNSW-NB15_2.csv': 'https://cloudstor.aarnet.edu.au/plus/s/2DhnLGDdEECo4ys/download?path=%2FUNSW-NB15%20-%20CSV%20Files&files=UNSW-NB15_2.csv',
    'UNSW-NB15_3.csv': 'https://cloudstor.aarnet.edu.au/plus/s/2DhnLGDdEECo4ys/download?path=%2FUNSW-NB15%20-%20CSV%20Files&files=UNSW-NB15_3.csv',
    'UNSW-NB15_4.csv': 'https://cloudstor.aarnet.edu.au/plus/s/2DhnLGDdEECo4ys/download?path=%2FUNSW-NB15%20-%20CSV%20Files&files=UNSW-NB15_4.csv',
    'NUSW-NB15_features.csv': 'https://cloudstor.aarnet.edu.au/plus/s/2DhnLGDdEECo4ys/download?path=%2FUNSW-NB15%20-%20CSV%20Files&files=NUSW-NB15_features.csv',
}

for filename, url in files.items():
    dest = f'{RAW}/{filename}'
    if os.path.exists(dest):
        print(f'Already exists, skipping: {filename}')
        continue
    print(f'Downloading {filename}...')
    try:
        urllib.request.urlretrieve(url, dest)
        size_mb = os.path.getsize(dest) / (1024 * 1024)
        print(f'  Done. Size: {size_mb:.1f} MB')
    except Exception as e:
        print(f'  Failed: {e}')

print('\nDownload step complete.')

  Failed: <urlopen error [Errno -5] No address associated with hostname>
  Failed: <urlopen error [Errno -5] No address associated with hostname>
  Failed: <urlopen error [Errno -5] No address associated with hostname>
  Failed: <urlopen error [Errno -5] No address associated with hostname>
  Failed: <urlopen error [Errno -5] No address associated with hostname>

Download step complete.


### Fallback: Manual Kaggle Download

If the UNSW direct links above fail (they sometimes do), use this Kaggle fallback.  
You need to upload your `kaggle.json` API key first.

In [6]:
# Only run this cell if Step 5 above failed

from google.colab import files
files.upload()  # upload your kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d dhoogla/unswnb15 -p {RAW} --unzip
print('Kaggle download complete.')

print('Fallback cell is commented out. Uncomment if Step 5 failed.')

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/dhoogla/unswnb15
License(s): CC-BY-NC-SA-4.0
100% 11.7M/11.7M [00:02<00:00, 4.88MB/s]

Kaggle download complete.
Fallback cell is commented out. Uncomment if Step 5 failed.


In [7]:
!kaggle datasets download -d mrwellsdavid/unsw-nb15 -p {RAW} --unzip

Dataset URL: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
License(s): unknown
100% 149M/149M [00:10<00:00, 14.9MB/s]



## Step 6: Merge CSV Files and Quick Sanity Check

In [8]:
import pandas as pd
import glob

col_names = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
    'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload',
    'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz',
    'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt',
    'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl',
    'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst',
    'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
    'ct_dst_src_ltm', 'attack_cat', 'label'
]

csv_files = sorted(glob.glob(f'{RAW}/UNSW-NB15_*.csv'))
print(f'Found {len(csv_files)} CSV files.')

dfs = []
for f in csv_files:
    df = pd.read_csv(f, header=None, names=col_names, low_memory=False)
    dfs.append(df)
    print(f'  Loaded {os.path.basename(f)}: {len(df):,} rows')

data = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows after merge: {len(data):,}')
print(f'Total columns: {len(data.columns)}')

Found 5 CSV files.
  Loaded UNSW-NB15_1.csv: 700,001 rows
  Loaded UNSW-NB15_2.csv: 700,001 rows
  Loaded UNSW-NB15_3.csv: 700,001 rows
  Loaded UNSW-NB15_4.csv: 440,044 rows
  Loaded UNSW-NB15_LIST_EVENTS.csv: 209 rows

Total rows after merge: 2,540,256
Total columns: 49


In [9]:
print('Shape:', data.shape)
print('\nColumn dtypes:')
print(data.dtypes)
print('\nFirst 3 rows:')
data.head(3)

Shape: (2540256, 49)

Column dtypes:
srcip                object
sport                object
dstip                object
dsport               object
proto                object
state                object
dur                 float64
sbytes              float64
dbytes              float64
sttl                float64
dttl                float64
sloss               float64
dloss               float64
service              object
Sload               float64
Dload               float64
Spkts               float64
Dpkts               float64
swin                float64
dwin                float64
stcpb               float64
dtcpb               float64
smeansz             float64
dmeansz             float64
trans_depth         float64
res_bdy_len         float64
Sjit                float64
Djit                float64
Stime               float64
Ltime               float64
Sintpkt             float64
Dintpkt             float64
tcprtt              float64
synack              float64
ackdat     

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132.0,164.0,31.0,...,0,3.0,7.0,1.0,3.0,1.0,1.0,1.0,NaN,0.0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528.0,304.0,31.0,...,0,2.0,4.0,2.0,3.0,1.0,1.0,2.0,NaN,0.0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146.0,178.0,31.0,...,0,12.0,8.0,1.0,2.0,2.0,1.0,1.0,NaN,0.0


In [10]:
print('Label distribution:')
print(data['label'].value_counts())
print('\nAttack category distribution:')
print(data['attack_cat'].value_counts())

Label distribution:
label
0.0    2218764
1.0     321283
Name: count, dtype: int64

Attack category distribution:
attack_cat
Generic             215481
Exploits             44525
 Fuzzers             19195
DoS                  16353
 Reconnaissance      12228
 Fuzzers              5051
Analysis              2677
Backdoor              1795
Reconnaissance        1759
 Shellcode            1288
Backdoors              534
Shellcode              223
Worms                  174
Name: count, dtype: int64


## Step 7: Save Merged Raw Dataset to Drive

In [11]:
raw_out = f'{BASE}/data/raw/UNSW_NB15_merged.csv'
data.to_csv(raw_out, index=False)
size_mb = os.path.getsize(raw_out) / (1024 * 1024)
print(f'Saved merged dataset to: {raw_out}')
print(f'File size: {size_mb:.1f} MB')

Saved merged dataset to: /content/drive/MyDrive/CSC5800_NIDS_Project/data/raw/UNSW_NB15_merged.csv
File size: 709.3 MB


## Step 8: Final Verification

In [12]:
print('=== Project Folder Structure ===')
for root, dirs, files_list in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    sub = '  ' * (level + 1)
    for f in files_list:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / (1024 * 1024)
        print(f'{sub}{f}  ({size:.1f} MB)')

print('\n=== Setup Complete ===')
print('You can now proceed to Notebook 2: Preprocessing and EDA')

=== Project Folder Structure ===
CSC5800_NIDS_Project/
  data/
    raw/
      UNSW_NB15_testing-set.parquet  (4.3 MB)
      UNSW_NB15_training-set.parquet  (9.2 MB)
      NUSW-NB15_features.csv  (0.0 MB)
      UNSW-NB15_1.csv  (161.2 MB)
      UNSW-NB15_2.csv  (157.6 MB)
      UNSW-NB15_3.csv  (147.4 MB)
      UNSW-NB15_4.csv  (93.1 MB)
      UNSW-NB15_LIST_EVENTS.csv  (0.0 MB)
      UNSW_NB15_testing-set.csv  (30.8 MB)
      UNSW_NB15_training-set.csv  (14.7 MB)
      UNSW_NB15_merged.csv  (709.3 MB)
    processed/
  models/
  results/
  figures/
  notebooks/

=== Setup Complete ===
You can now proceed to Notebook 2: Preprocessing and EDA
